In [ ]:
# Install Python packages
!pip install -q streamlit langchain-google-genai langchain-experimental pandas tabulate

# Install localtunnel to expose the Streamlit port
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 3s
⠋
⠋3 packages are looking for funding
⠋  run 

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_experimental.agents import create_pandas_dataframe_agent

# --- Page Configuration ---
st.set_page_config(
    page_title="Internal CSV FAQ Agent",
    page_icon="📂",
    layout="wide"
)

st.title("📂 Multi-Domain CSV FAQ Assistant")
st.caption("Upload company policy/FAQ CSVs and get instant, grounded answers ready for customer support.")

# --- Sidebar: Configuration & Uploads ---
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input("Enter Gemini API Key", type="password", help="Get your key from Google AI Studio")

    st.header("📄 Upload Data")
    uploaded_files = st.file_uploader(
        "Upload one or more CSV files",
        type=["csv"],
        accept_multiple_files=True
    )

# --- Data Loading & Preview ---
dfs = []
file_names = []

if uploaded_files:
    st.subheader("📋 Uploaded Data Previews")
    preview_tabs = st.tabs([file.name for file in uploaded_files])

    for idx, file in enumerate(uploaded_files):
        try:
            df = pd.read_csv(file)
            dfs.append(df)
            file_names.append(file.name)
            with preview_tabs[idx]:
                st.write(f"**Shape:** {df.shape[0]} rows, {df.shape[1]} columns")
                #st.dataframe(df.head(5), use_container_width=True)
                st.table(df.head(5))
        except Exception as e:
            st.error(f"Error reading `{file.name}`: {e}")

# --- Question & Answer Interface ---
st.divider()
st.subheader("💬 Ask a Question")

# Example question pills for quick testing
example_prompts = [
    "What is the return policy for electronics?",
    "What does the extended warranty cover?",
    "What are the visiting hours in the hospital?",
    "What is the API rate limit for the free plan?"
]

selected_example = st.pills("Quick Examples:", example_prompts)
user_query = st.text_input("Enter customer question:", value=selected_example if selected_example else "")

if st.button("Generate Answer", type="primary"):
    if not api_key:
        st.warning("⚠️ Please provide a Gemini API Key in the sidebar.")
    elif not dfs:
        st.warning("⚠️ Please upload at least one CSV file.")
    elif not user_query.strip():
        st.warning("⚠️ Please enter a question.")
    else:
        with st.spinner("Searching CSV data..."):
            try:
                # 1. Initialize Gemini LLM with zero temperature for factual accuracy
                llm = ChatGoogleGenerativeAI(
                    model="gemini-3.6-flash",
                    google_api_key=api_key,
                    temperature=0.0
                )

                # 2. Strict grounding instructions
                prefix_prompt = """
                You are a factual, internal customer support assistant for a business.
                Your task is to answer questions strictly using the provided pandas DataFrame(s).

                CRITICAL RULES:
                1. You must ONLY use facts and numbers present in the uploaded tables.
                2. Do NOT use any external or general world knowledge.
                3. If the answer cannot be found directly in or computed from the data, you MUST reply:
                   "I could not find this information in the uploaded files."
                4. For text inquiries, extract and summarize the exact policy or answer.
                5. For numeric inquiries, calculate exact figures (e.g., sums, averages, counts).
                6. Keep the tone professional, clear, and ready to be copy-pasted into customer emails or chats.
                """

                # 3. Create Pandas DataFrame Agent
                agent = create_pandas_dataframe_agent(
                    llm=llm,
                    df=dfs if len(dfs) > 1 else dfs[0],
                    verbose=True,
                    allow_dangerous_code=True,
                    prefix=prefix_prompt,
                    agent_type="tool-calling"
                )

                # 4. Run the Agent
                response = agent.invoke({"input": user_query})
                output_text = response["output"]

                # 5. Display Answer
                st.success("### Ready-to-Send Response:")
                st.code(output_text, language="markdown")
                st.info("💡 You can click the copy icon on the top-right of the box above to paste directly into customer chats.")

            except Exception as err:
                st.error(f"Error processing query: {err}")

Overwriting app.py


In [ ]:
import urllib.request
import time
import subprocess

# 1. Start Streamlit in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

# 2. Give the server a few seconds to boot
time.sleep(3)

# 3. Fetch your Google Colab external IP (needed as the Tunnel Password)
colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

print("=" * 60)
print(f"🔑 YOUR TUNNEL PASSWORD (EXTERNAL IP): {colab_ip}")
print("=" * 60)
print("Click the link below, paste the IP above into the 'Endpoint IP' field, and click Submit:")
print("=" * 60)

# 4. Launch LocalTunnel
!npx localtunnel --port 8501

🔑 YOUR TUNNEL PASSWORD (EXTERNAL IP): 34.66.140.85
Click the link below, paste the IP above into the 'Endpoint IP' field, and click Submit:
⠙⠹⠸⠼⠴⠦your url is: https://twenty-wasps-tap.loca.lt
^C
